## Setup
- download [dataset](https://www.kaggle.com/competitions/dfl-bundesliga-data-shootout/data) using [Kaggle API](https://github.com/Kaggle/kaggle-api), 150 videos that we can use 
for training
- [player dataset](https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc)
- pip install ultralytics to get access to YOLO https://docs.ultralytics.com/quickstart/

## Libaries

In [5]:
import os
import cv2
from ultralytics import YOLO

## Load Model

In [2]:
#define the YOLO model
model = YOLO('yolov8x')

## Object Detection

### Inference on video clip
The video file is passed to the model frame by frame where objects are detected, boundary boxes are drawn and an output video file is created with the overlay.

### Detected Objects

YOLOv8 is capable of detecting 80 different objects in images and video clips. For this task I am just looking for two classes, a `Person` and a `Sports Ball`. The full list of classes however can be seen below:

| Index |     Object     | Index |     Object     | Index |     Object     | Index |     Object     |
|:-----:|:--------------:|:-----:|:--------------:|:-----:|:--------------:|:-----:|:--------------:|
|   **0**   |    **person**      |  20   |    elephant    |  40   |   wine glass   |  60   |  dining table  |
|   1   |    bicycle     |  21   |     bear       |  41   |      cup       |  61   |     toilet     |
|   2   |      car       |  22   |     zebra      |  42   |      fork      |  62   |       tv       |
|   3   |  motorcycle   |  23   |    giraffe     |  43   |     knife      |  63   |     laptop     |
|   4   |   airplane     |  24   |   backpack     |  44   |     spoon      |  64   |     mouse      |
|   5   |      bus       |  25   |    umbrella    |  45   |      bowl      |  65   |     remote     |
|   6   |     train      |  26   |    handbag     |  46   |    banana      |  66   |   keyboard     |
|   7   |     truck      |  27   |      tie       |  47   |     apple      |  67   |  cell phone    |
|   8   |     boat       |  28   |    suitcase    |  48   |   sandwich     |  68   |   microwave    |
|   9   | traffic light  |  29   |    frisbee     |  49   |    orange      |  69   |     oven       |
|  10   | fire hydrant   |  30   |     skis       |  50   |   broccoli     |  70   |    toaster     |
|  11   |   stop sign    |  31   |   snowboard    |  51   |    carrot      |  71   |      sink      |
|  12   | parking meter  |  **32**   |  **sports ball**   |  52   |    hot dog     |  72   | refrigerator   |
|  13   |     bench      |  33   |     kite       |  53   |     pizza      |  73   |     book       |
|  14   |      bird      |  34   | baseball bat   |  54   |     donut      |  74   |     clock      |
|  15   |      cat       |  35   | baseball glove |  55   |     cake       |  75   |     vase       |
|  16   |      dog       |  36   |   skateboard   |  56   |     chair      |  76   |    scissors    |
|  17   |     horse      |  37   |    surfboard   |  57   |     couch      |  77   |  teddy bear    |
|  18   |     sheep      |  38   |  tennis racket |  58   | potted plant   |  78   |   hair drier   |
|  19   |      cow       |  39   |     bottle     |  59   |      bed       |  79   |   toothbrush   |


We can loop through the results to see what was detected. These results, provide comprehensive information about detected objects, their locations, classes, and confidence scores. `cls: tensor([0.])` is a `Person`, `cls: tensor([32.])` is the sports ball. Approximately 20 people were found in the video clip and 1 ball. However ball tracking is not very good, the model is detecting both players on pitch and spectators off the pitch. Also we are not using colours to distingish teams or the referee.

![Detected Objects](https://raw.githubusercontent.com/rob-sullivan/ai/football-tracking/football-tracking/detected_objects.png)


### Improved Object Tracking
By using [ByteTrack](https://github.com/ifzhang/ByteTrack.git) we can improve [multi-object tracking](https://github.com/ultralytics/ultralytics/tree/main/ultralytics/trackers), since it employs a novel technique to instead ignoring low threhold objects, track their similar trajectories across frames.

The following was done to work with ByteTrack:
* https://www.youtube.com/watch?v=6LGpf-a1K1Q
* https://youtu.be/LNwODJXcvt4
* Track using ByteTrack tracker
```shell
    > yolo track model=path/to/best.pt tracker="bytetrack.yaml"
```

## Object Tracking
ref: https://www.youtube.com/watch?v=uMzOcCNKr5A&t=644s

### Load Video & Read Frames

In [ ]:
%%capture
#capture stops any text output in this cell

# Load video file
video_path = './data/08fd33_4.mp4'
cap = cv2.VideoCapture(video_path)

# Get original video properties
fps = cap.get(cv2.CAP_PROP_FPS)  # Frames per second of the original video
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))  # Original width
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))  # Original height

# Define the codec and create VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'XVID')  # Codec for output video
# Set the output size to 50% of the original video size
output_width = int(frame_width * 0.5)
output_height = int(frame_height * 0.5)
out = cv2.VideoWriter('./data/08fd33_4_tracked.mp4', fourcc, fps, (output_width, output_height))

# Initialize detection model
initial_detection = True
tracker = None

# Processing each frame
while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Track objects in the current frame
    results = model.track(frame, persist=True, tracker="bytetrack.yaml")
    
    # Plot results on the frame (with bounding boxes/tracks)
    frame_ = results[0].plot()
    
    # Resize the frame (downscale to 50% of the original size)
    resized_frame_ = cv2.resize(frame_, (output_width, output_height), interpolation=cv2.INTER_AREA)
    
    # Write the resized frame to the output video file
    out.write(resized_frame_)

    # Optional: Display the frame (remove if you only want to save the video)
    # cv2.imshow('Football Tracker', resized_frame_)
    # if cv2.waitKey(25) & 0xFF == ord('q'):
    #     break

# Release resources
cap.release()
out.release()
cv2.destroyAllWindows()

![Tracked Objects](https://raw.githubusercontent.com/rob-sullivan/ai/football-tracking/football-tracking/tracked_objects.png)

After tracking objects across frames using bytetrack I found that the ball was not detected most of the time. To solve this issue I fine tune the YOLOv8 model on labelled examples of the sports ball to improve ball detection. To label data I used the example mp4 video file with [Label Studio](https://github.com/HumanSignal/label-studio).

To begin I needed examples of images that I can label so I converted mp4 to a folder of images.

In [6]:
# paths
video_path = './data/08fd33_4.mp4'
output_folder = './data/football_images'

# Create folder if not there
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Open the video file
cap = cv2.VideoCapture(video_path)

# set counter to zero
frame_number = 0

# Loop frames
while True:
    # Read a frame from the video
    ret, frame = cap.read()
    
    # If there are no more frames, break the loop
    if not ret:
        break
    
    # Construct the filename for each frame
    frame_filename = os.path.join(output_folder, f'frame_{frame_number:04d}.png')
    
    # Save the frame as an image
    cv2.imwrite(frame_filename, frame)
    
    # Increment the frame counter
    frame_number += 1

# Release the video capture object
cap.release()

print(f"All frames extracted and saved to {output_folder}")

All frames extracted and saved to ./data/football_images


With a 100 images I began labelling the ball building up a training set to fine tune YOLO

![Tracked Objects](https://raw.githubusercontent.com/rob-sullivan/ai/football-tracking/football-tracking/label_studio.png)